# Student Performance Tracker
### Final Grade Prediction using Multiple Linear Regression

**Project objective:** Explore student academic data and train a supervised machine-learning regression model to predict the final grade (`G3`) from selected student attributes and earlier-period grades.

This notebook covers:
1. Dataset loading and inspection
2. Exploratory data analysis (EDA)
3. Feature and target selection
4. Train-test splitting
5. Categorical feature encoding
6. Multiple Linear Regression
7. Model evaluation
8. Actual-versus-predicted and residual visualizations
9. Prediction for a new student

> **Model scope:** This is Model A – Final Grade Prediction. It uses `G1` and `G2`, so it predicts `G3` when those earlier grades are already available. It is not an early-warning model that predicts risk before the first or second period grades are known.

## 1. Import libraries

We use:
- **Pandas** for loading and manipulating tabular data.
- **NumPy** for numerical operations.
- **Matplotlib** for visualizations.
- **Scikit-learn** for splitting data, preprocessing categorical features, training the regression model, and evaluating predictions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load the dataset

The notebook first looks for the dataset in the `data` folder beside this notebook. It also checks the notebook folder itself.

For this project, use the UCI Portuguese student performance file, commonly named **`student-por.csv`**. The file is semicolon-separated, so we specify `sep=';'`.

**GitHub folder suggestion:**
```text
Student-Performance-Tracker/
│
├── main.ipynb
├── README.md
└── data/
    └── student-por.csv
```

If you store the CSV somewhere else, update `DATA_PATH` below.

In [ ]:

DATA_PATH = "data/student_data_csv"

data = pd.read_csv(DATA_PATH, sep=";")
print("Dataset loaded successfully!")

## 3. Initial data inspection

Before analysis, inspect the first records, dataset dimensions, column types, missing values, duplicate rows, and descriptive statistics.

- `head()` displays the first five rows.
- `info()` shows column names, data types, and non-null counts.
- `shape` returns the number of rows and columns.
- `isnull().sum()` counts missing values in each column.
- `duplicated().sum()` counts duplicate rows.
- `describe()` summarizes numerical columns.

In [ ]:
display(data.head())

print("\nDataset information:")
data.info()

print("\nDataset shape:", data.shape)

print("\nMissing values in each column:")
print(data.isnull().sum())

print("\nNumber of duplicate rows:", data.duplicated().sum())

print("\nDescriptive statistics:")
display(data.describe())

## 4. Exploratory Data Analysis (EDA)

### 4.1 Distribution of final grades (`G3`)

A histogram shows how final grades are distributed. It helps identify common grade ranges, spread, and possible skewness.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(data["G3"], bins=15, edgecolor="black")
plt.xlabel("Final Grade (G3)")
plt.ylabel("Number of Students")
plt.title("Distribution of Final Grades")
plt.tight_layout()
plt.show()

### 4.2 Correlation analysis

Correlation measures the strength and direction of a linear relationship between numerical variables. Values range from `-1` to `+1`.

- Positive correlation: both variables tend to increase together.
- Negative correlation: one tends to decrease as the other increases.
- Correlation does not prove causation.

In [ ]:
correlation_columns = ["G3", "G2", "G1", "studytime", "absences"]
correlation_matrix = data[correlation_columns].corr()

print("Correlation Matrix:")
display(correlation_matrix)

plt.figure(figsize=(7, 5))
plt.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(correlation_columns)), correlation_columns, rotation=45)
plt.yticks(range(len(correlation_columns)), correlation_columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

### 4.3 Relationship between first-period and final grades

A scatter plot helps visualize the relationship between `G1` and `G3`. Each point represents one student.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(data["G1"], data["G3"], alpha=0.6)
plt.xlabel("First Period Grade (G1)")
plt.ylabel("Final Grade (G3)")
plt.title("First Period Grade vs Final Grade")
plt.tight_layout()
plt.show()

### 4.4 Descriptive at-risk grouping

For this analysis, a student is grouped as **at risk** when `G3 < 10`.

This creates a Boolean analysis column. It is a retrospective grouping based on the known final grade, **not a prediction produced by the regression model**.

In [ ]:
data["At_Risk_Students"] = data["G3"] < 10

at_risk_count = int(data["At_Risk_Students"].sum())
total_students = len(data)
at_risk_percentage = (at_risk_count / total_students) * 100

print("At-risk students:", at_risk_count)
print(f"At-risk percentage: {at_risk_percentage:.2f}%")

## 5. Feature selection and target variable

The model uses the following input features:

| Feature | Meaning |
|---|---|
| `G1` | First-period grade |
| `G2` | Second-period grade |
| `studytime` | Weekly study-time category |
| `absences` | Number of school absences |
| `failures` | Number of past class failures |
| `school` | Student's school (categorical) |
| `sex` | Student's sex (categorical) |

**Target (`y`):** `G3`, the final grade.

`X` contains the information used to make predictions. `y` contains the value the model learns to predict.

In [ ]:
feature_columns = [
    "G1",
    "G2",
    "studytime",
    "absences",
    "failures",
    "school",
    "sex"
]

X = data[feature_columns]
y = data["G3"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

## 6. Split the data into training and testing sets

We reserve 20% of the records for testing and use 80% for training.

- **Training set:** used to learn the model.
- **Testing set:** used to evaluate the model on records it did not train on.
- `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 7. Preprocess categorical and numerical features

A machine-learning model requires numerical input.

- Numerical features (`G1`, `G2`, `studytime`, `absences`, `failures`) are passed through unchanged.
- Categorical features (`school`, `sex`) are converted into numerical indicator columns using **One-Hot Encoding**.

`handle_unknown="ignore"` prevents an error if a new category appears when transforming test or future records.

### Important: fit only on training data

We use `fit_transform()` on training data to learn the category mapping and transform the training records. We use only `transform()` on test data so it uses the exact same learned mapping.

This avoids fitting preprocessing separately on the test set.

In [ ]:
numerical_features = [
    "G1",
    "G2",
    "studytime",
    "absences",
    "failures"
]

categorical_features = [
    "school",
    "sex"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

## 8. Train the Multiple Linear Regression model

Multiple Linear Regression estimates a continuous target using multiple input features.

The general hypothesis is:

\[
\hat{y} = \theta_0 + \theta_1x_1 + \theta_2x_2 + \cdots + \theta_nx_n
\]

Here:
- `ŷ` is the predicted final grade.
- `θ₀` is the intercept.
- `θ₁ ... θₙ` are learned coefficients.
- `x₁ ... xₙ` are the processed input features.

Scikit-learn's `LinearRegression` learns coefficients by solving a least-squares problem.

In [ ]:
model = LinearRegression()

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

print("Model training completed.")

## 9. Evaluate the model

We use three evaluation metrics.

### Mean Absolute Error (MAE)
Average absolute difference between actual and predicted grades. Lower is better.

### Root Mean Squared Error (RMSE)
Square root of the average squared prediction error. It penalizes larger errors more strongly than MAE. It is expressed in grade points.

### R-squared (R²)
Measures how much of the variation in the test-set target values is explained by the model relative to a mean-baseline model. A value closer to 1 is generally better; it can also be negative.

In [ ]:
MAE = mean_absolute_error(y_test, y_pred)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {MAE:.3f}")
print(f"Root Mean Squared Error (RMSE): {RMSE:.3f}")
print(f"R-squared (R²): {R2:.3f}")

## 10. Compare actual and predicted grades

The table compares actual test-set grades with the corresponding model predictions. This is a spot-check; the evaluation metrics above summarize performance across the full test set.

In [ ]:
comparison = pd.DataFrame({
    "Actual Grade": y_test.to_numpy(),
    "Predicted Grade": y_pred
})

comparison["Prediction Error"] = (
    comparison["Actual Grade"] - comparison["Predicted Grade"]
)

display(comparison.head(10))

### 10.1 Actual vs predicted visualization

Points close to the diagonal reference line indicate predictions close to the actual grades.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.6)

grade_min = min(float(y_test.min()), float(np.min(y_pred)))
grade_max = max(float(y_test.max()), float(np.max(y_pred)))

plt.plot(
    [grade_min, grade_max],
    [grade_min, grade_max],
    linestyle="--"
)

plt.xlabel("Actual Grades")
plt.ylabel("Predicted Grades")
plt.title("Actual vs Predicted Final Grades")
plt.tight_layout()
plt.show()

### 10.2 Residual analysis

A residual is the difference between an actual value and its prediction:

\[
\text{Residual} = y_{actual} - y_{predicted}
\]

A residual plot can help identify patterns that the model has not captured. Ideally, residuals should be scattered around zero without a clear systematic pattern.

In [ ]:
residuals = y_test.to_numpy() - y_pred

plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Grades")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.tight_layout()
plt.show()

## 11. Predict the final grade for a new student

Create a DataFrame containing the same feature names used during training.

**Important:** use the already-fitted preprocessor to transform new data. Do not fit a new encoder on the new student record.

In [ ]:
student = pd.DataFrame([{
    "G1": 15,
    "G2": 16,
    "studytime": 2,
    "absences": 3,
    "failures": 0,
    "school": "GP",
    "sex": "F"
}])

student_processed = preprocessor.transform(student)
predicted_grade = model.predict(student_processed)

print(
    "Predicted Final Grade:",
    round(float(predicted_grade[0]), 2)
)

## 12. Inspect the prediction range

This shows the minimum and maximum values predicted for the test set. It is a descriptive check, not an accuracy metric.

Linear Regression is not constrained to output only grades from 0 to 20, so predictions outside the valid grade range should be inspected and discussed rather than silently clipped.

In [ ]:
minimum_prediction = np.min(y_pred)
maximum_prediction = np.max(y_pred)

print(f"Minimum predicted grade: {minimum_prediction:.3f}")
print(f"Maximum predicted grade: {maximum_prediction:.3f}")

## 13. Understanding the connection to Gradient Descent

In the separate Gradient Descent implementation, we used simple linear regression:

\[
h_\theta(x) = \theta_0 + \theta_1x
\]

There was one input feature and two parameters (`θ₀`, `θ₁`). We calculated the cost and partial derivatives, then updated the parameters iteratively:

\[
\theta_j := \theta_j - \alpha\frac{\partial J}{\partial\theta_j}
\]

This Student Performance Tracker extends the linear hypothesis to multiple processed features:

\[
\hat{y} = \theta_0 + \theta_1x_1 + \cdots + \theta_nx_n
\]

The underlying idea is connected: choose coefficients to minimize prediction error. However, **this notebook currently uses Scikit-learn's `LinearRegression`, not the custom Gradient Descent code**. The gradient descent project demonstrates the optimization process manually; this notebook applies a library implementation to a real multi-feature dataset.

## 14. Conclusion

This project demonstrates a complete supervised regression workflow:
- Loading and inspecting student data
- Performing exploratory data analysis
- Selecting input features and a target
- Splitting data into training and testing sets
- Encoding categorical variables
- Training a Multiple Linear Regression model
- Evaluating predictions using MAE, RMSE, and R²
- Visualizing predictions and residuals
- Predicting the final grade for a new student

### Limitations
1. The model uses `G1` and `G2`; therefore, it is not an early-warning predictor before those grades are available.
2. The dataset contains records from Portuguese secondary schools, so results may not generalize to other institutions.
3. Linear Regression may predict values outside the valid grade range.
4. A high test score does not prove that the model will perform equally well on a different population.

### Future scope
A separate early-warning classification model could be developed later, using only features available at an earlier stage. That is outside the scope of this Model A notebook.

### Dataset reference
Cortez, P. and Silva, A. M. G. (2008), *Using Data Mining to Predict Secondary School Student Performance*, Proceedings of the 5th Annual Future Business Technology Conference.

UCI Machine Learning Repository: [Student Performance Dataset](https://archive.ics.uci.edu/dataset/320/student+performance)